# Dated payments: from a caplet to an amortizing note

**Question.** Can explicit payoff events reproduce a native note's coupon and
principal economics, without confusing representation, calibration, and sampling?

We use the illustrative 24 February 2026 USD/SOFR curve, a calibrated Gaussian
rate model, and a six-year semiannual amortizing underlying. The fixed coupon
uses Actual/365, floating periods Actual/360; dates are unadjusted. Issuer
exercise is deliberately omitted. The calibrated main examples use nonzero
volatility; a separate zero-volatility experiment isolates representation.
[Setup](../README.md) · [Callable note](../valuation/bonds/stepped_coupon_bond.ipynb).

`ScriptedProduct` supplies dated observations and payments, not a Python
pricing callback. The native engine returns numeric variable values. Only the
selected `result_variable` receives the fitted martingale-control adjustment;
other returned variables are not automatically present values.


In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market, load_pricers
from stepped_cashflows import underlying_script

market = load_market()
ref = market.quotes.reference_date
pricers = load_pricers("bond_tree.json")


## One event observes a rate and schedules a payment

An event runs on its `date`. `RateIndex` observes the specified accrual-period
rate from the model at that event; `cvg` computes a year fraction; `pays` adds
a discounted payment to a named accumulator. Here `note` is the PV accumulator,
while `rate` and `accrual` are intermediate numeric variables.

The `result_variable` names the engine's target variable; it does not filter
the returned dictionary to only that key. `evaluate_script` currently returns
values, not Python sensitivity or cashflow tables. The model is resolved from
the context configuration below, so there is no implicit zero-volatility model.

In [2]:
fixing = ref + "1Y"
payment = fixing + "3M"
scripted_caplet = qs.ScriptedProduct.from_dict(
    {
        "name": "SCRIPTED-CAPLET",
        "currency": qs.Currency.USD,
        "result_variable": "note",
        "events": [
            {
                "date": fixing,
                "code": (
                    f'note = 0; rate = RateIndex("SOFR", "{fixing}", "{payment}"); '
                    f'accrual = cvg("{fixing}", "{payment}", "Actual360"); '
                    f'note pays 1000000 * max(rate - 0.045, 0) * accrual on "{payment}";'
                ),
            }
        ],
    }
)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)

with context:
    values = context.evaluate_script(
        scripted_caplet,
        n_paths=2048,
        seed=42,
    )

pd.Series(values)


note       1137.669549
accrual       0.247222
rate          0.042082
dtype: float64

This uses the configured Hull–White model's equivalent LGM representation.
It need not exactly equal a Black caplet priced at an arbitrary smile point:
the model is fitted to an ATM calibration basket and Monte Carlo adds sampling
error. The shared bond configuration selects 3M, 6M, 1Y, 2Y, 3Y and 5Y
ATM expiries. The selected basket and its domain should be checked when inputs change; this surface quotes strikes from 3.5% to 5.5%. The final calibrated sigma
segment continues beyond 5Y for the six-year underlying below. Keep model
choice distinct from payoff representation.

## Reuse the stepped note's underlying payments

The contract below matches the six-year coupon and principal schedule in the
[callable-bond example](../valuation/bonds/stepped_coupon_bond.ipynb), with issuer calls omitted.
`underlying_script` translates this specific semiannual example to dated
events. It is intentionally not a replacement for the library's general
schedule construction. Its implementation is short enough to inspect in
[stepped_cashflows.py](stepped_cashflows.py).


In [3]:
underlying = qs.SteppedCouponBond(
    identifier="STEPPED-UNDERLYING",
    start_date=ref,
    maturity_date=ref + "6Y",
    notional=1_000_000,
    currency=qs.Currency.USD,
    payment_frequency=qs.Frequency.Semiannual,
    coupon_day_counter=qs.DayCounter.Actual365,
    coupons=[
        qs.FixedRateCoupon(rate=qs.InterestRate(rate=0.045)),
        qs.OptionEmbeddedCoupon(
            index=qs.MarketIndex.SOFR,
            payoff=(qs.PayoffOps.Index + 0.006).max(0.0).min(0.06),
        ),
    ],
    notional_percentages=[1.0, 0.90, 0.75],
    call_schedule=[],
)
script = underlying_script(underlying)
pd.DataFrame(script.to_dict()["events"]).head(3)

,date,code
0,2026-02-24,"bond = 0; accrual = cvg(""2026-02-24"", ""2026-08..."
1,2026-08-24,"accrual = cvg(""2026-08-24"", ""2027-02-24"", ""Act..."
2,2027-02-24,"accrual = cvg(""2027-02-24"", ""2027-08-24"", ""Act..."


The first event initializes `bond`, fixes the first coupon, and schedules its
coupon plus principal reduction. Later events observe SOFR, apply spread and
cap/floor, and schedule their payments. The final payment returns all remaining
principal. The helper uses Actual/365 for the fixed coupon and SOFR's
Actual/360 for floating coupons.

This is the **non-callable underlying at zero spread**. The absence of exercise
logic is an economic difference from the callable note, not a numerical error.

In [4]:
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces,
    pricer_configurations=pricers,
)

with context:
    stochastic = context.evaluate_script(
        script,
        n_paths=4096,
        seed=42,
    )
print(f"Calibrated scripted underlying PV: USD {stochastic['bond']:,.2f}")


Calibrated scripted underlying PV: USD 1,010,885.99


## Isolate a representation check

To check that the two representations describe the same coupon and amortization
payments, use a **separate zero-volatility context**, remove issuer exercise,
and set zero Z-spread. The path count can be one because this check is
deterministic. This does not validate stochastic calibration or issuer-call
exercise, and it does not change the calibrated examples above.

The only model change is explicit below. A 0.05% PV tolerance accommodates the
native tree's time discretization for the floating coupons.

In [5]:
deterministic_spec = pricers[0].to_dict()
deterministic_spec["models"][0]["model"]["HullWhite"]["parameter_source"] = {
    "Fixed": {"sigma": 0.0}
}
deterministic_pricer = qs.PricerConfiguration.from_dict(deterministic_spec)
context = qs.PricingContext(
    quotes=market.quotes,
    curves=market.curves,
    csa_policy=market.csa_policy,
    pricer_configurations=[deterministic_pricer],
)

with context:
    native = context.evaluate(
        qs.Trade(underlying, ref, pricing_anchor=qs.BondPricingAnchor.z_spread(0.0)),
        [qs.Request.Value],
    )
    scripted = context.evaluate_script(
        script,
        n_paths=1,
        seed=42,
    )

pd.Series(
    {
        "native_pv_usd": native.price,
        "scripted_pv_usd": scripted["bond"],
        "difference_usd": native.price - scripted["bond"],
    }
)

native_pv_usd      1.020546e+06
scripted_pv_usd    1.020546e+06
difference_usd     0.000000e+00
dtype: float64

## Verify representation without overstating Monte Carlo equivalence

The deterministic native/script comparison holds coupons, amortization,
discount curve, zero spread, and zero volatility constant. Its tolerance is
0.05% of native PV to accommodate the tree's floating-coupon discretization;
this is not a tolerance for the main stochastic model.

At finite paths the selected scripted result may include a martingale-control
adjustment. Raw expected payment PVs, if obtained through lower-level APIs,
need not sum to that adjusted result. Likewise, a variable such as `rate`
is not a cashflow accumulator. We assess a stochastic run using repeated
seeds, not an exact equality to an off-ATM Black price under a different model.


In [6]:
import math

assert abs(native.price - scripted["bond"]) <= 0.0005 * abs(native.price)
assert len(script.to_dict()["events"]) == 12
assert values["note"] > 0 and all(math.isfinite(value) for value in values.values())
with qs.PricingContext(
    quotes=market.quotes, curves=market.curves, csa_policy=market.csa_policy,
    volatility_surfaces=market.volatility_surfaces, pricer_configurations=pricers,
) as context:
    estimates = [
        {
            "seed": seed,
            "caplet_pv_usd": context.evaluate_script(
                scripted_caplet, n_paths=2048, seed=seed
            )["note"],
        }
        for seed in [41, 42, 43, 44, 45]
    ]
estimates = pd.DataFrame(estimates)
assert math.isclose(
    estimates.loc[estimates["seed"] == 42, "caplet_pv_usd"].iloc[0],
    values["note"], abs_tol=1e-10,
)
print(f"Deterministic representation difference: USD {native.price - scripted['bond']:.6f}.")
print(f"Across-seed SD of 2,048-path caplet estimates: USD {estimates['caplet_pv_usd'].std():.4f}.")
estimates


Deterministic representation difference: USD 0.000000.
Across-seed SD of 2,048-path caplet estimates: USD 3.4266.


,seed,caplet_pv_usd
0,41,1136.335996
1,42,1137.669549
2,43,1129.344025
3,44,1137.054234
4,45,1136.635810


## Conclusion

The deterministic experiment verifies payment representation at matching model
assumptions. The calibrated runs answer a different question: model-based
option and underlying values, with finite-sample uncertainty. Removing calls
does not implement issuer optimization; the script is not a second price of the
callable note. The Python bridge currently returns variable values, not raw
payoff paths, cashflow tables, custom correlations, or scripted sensitivity
tables. The across-seed dispersion is a diagnostic, not a formal pathwise
standard-error estimate.


Keep the comparison at matching terms, spread, and model assumptions when
adapting this example. `evaluate_script` currently assembles rate models; it
does not expose arbitrary FX/equity factors, custom correlations, raw payoff
paths, or sensitivity tables through this Python entry point.